# §2.2.4 — 이분산 회귀: $\sigma(x)$ 를 함께 학습하면

> 딥러닝 교재 · 1부 2장 2절 4항 (🐍)
> 선행: §2.2.1(제곱오차 = 등분산 가우시안) · §2.2.2(로그 분산 항) · §2.2.3(MSE vs MAE)

## 이 노트북이 답하는 질문

1. **$\sigma(x)$ 를 함께 학습하면 정말 나아지는가?** 등분산 모형과 시험 NLL을 비교한다.
2. **$\hat\sigma(x)$ 가 참 $\sigma(x)$ 를 회복하는가?** 참값을 아는 합성 자료이므로 직접 채점할 수 있다.
3. **§2.2.2 4절이 유도한 실패 기제가 실제로 나타나는가?** 모형이 어려운 영역에서 $\sigma$ 를 키워 손실을 회피하는지 본다.
4. **그 실패를 어떻게 진단하는가?** $\hat\sigma$ 가 크다는 것이 "자료가 시끄럽다"인지 "모형이 못 맞힌다"인지 구별할 수 있는가.

**예상 실행 시간** CPU 단일 코어 약 70초 (`FAST = True`이면 약 25초).

**설계의 핵심은 두 영역을 갈라 놓은 것이다** — 평균이 맞히기 어려운 곳과 잡음이 실제로 큰 곳을
서로 다른 $x$ 에 두었다. 그래야 $\hat\sigma$ 가 둘을 혼동하는지 판정할 수 있다.

---
## 0. 설정

In [ ]:
import os, glob, time
import numpy as np
import matplotlib.pyplot as plt

_t0 = time.time()

# ── 손잡이 (마지막 셀에 전체 목록) ──────────────────────────
FAST     = False
SEED     = 20260806
N_TRAIN  = 1500
STEPS    = 3000
WIDTH    = 64
S_CLAMP  = (-8.0, 8.0)   # log σ² 의 절단 범위 (§2.2.2 5절)
SAVE_PDF = False
FIG_DIR  = 'figs'
# ──────────────────────────────────────────────────────────
if FAST:
    N_TRAIN, STEPS = 800, 1500

CB = ['#000000','#E69F00','#56B4E9','#009E73','#D55E00','#0072B2','#CC79A7','#F0E442']
plt.rcParams.update({'figure.dpi':120,'font.size':10,'axes.grid':True,'grid.alpha':0.3,
                     'axes.prop_cycle':plt.cycler(color=CB),'figure.autolayout':True})
import matplotlib.font_manager as fm
for _p in glob.glob('/usr/share/fonts/**/*CJK*.ttc', recursive=True)[:6]:
    try:
        fm.fontManager.addfont(_p)
    except Exception:
        pass
_av = {f.name for f in fm.fontManager.ttflist}
KO_FONT = next((f for f in ['NanumGothic','Malgun Gothic','AppleGothic','Noto Sans CJK KR',
                            'Noto Sans KR','NanumBarunGothic','Noto Sans CJK JP'] if f in _av), None)
if KO_FONT:
    plt.rcParams['font.family'] = KO_FONT
    plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42
def lab(ko, en):
    return ko if KO_FONT else en
_fi=[0]
def show(name):
    _fi[0] += 1
    if SAVE_PDF:
        os.makedirs(FIG_DIR, exist_ok=True)
        plt.savefig(os.path.join(FIG_DIR, f'fig_2_2_4_{_fi[0]}_{name}.pdf'),
                    bbox_inches='tight', pad_inches=0.02)
    plt.show()

print(f"numpy {np.__version__} | FAST={FAST} | 한글폰트: {KO_FONT or '없음(영문 라벨)'}")

---
## 1. 자료 — 두 어려움을 서로 다른 곳에 둔다

$$\mu(x) = \sin x + 0.3x + 0.8\,\sin(10x)\,e^{-(x+1.5)^2/0.5}, \qquad
\sigma(x) = 0.08 + 0.55\,e^{-(x-1.5)^2/0.5}$$

| 위치 | 무엇이 어려운가 | 참 $\sigma$ |
|---|---|---|
| $x \approx -1.5$ | **평균이 고주파**라 맞히기 어렵다 | **작다** (0.08) |
| $x \approx +1.5$ | 평균은 매끄럽다 | **크다** (0.63) |

정직한 $\hat\sigma(x)$ 라면 오른쪽에서만 커야 한다. **왼쪽에서도 커진다면 모형 오차를 자료 잡음으로 보고한 것**이다.

In [ ]:
def mu_true(x):
    x = np.asarray(x, float)
    return np.sin(x) + 0.3*x + 0.8*np.sin(10*x)*np.exp(-(x+1.5)**2/0.5)

def sig_true(x):
    x = np.asarray(x, float)
    return 0.08 + 0.55*np.exp(-(x-1.5)**2/0.5)

def sample(n, g):
    x = g.uniform(-3, 3, n)
    return x, mu_true(x) + sig_true(x)*g.normal(size=n)

x_tr, y_tr = sample(N_TRAIN, np.random.default_rng(SEED))
x_te, y_te = sample(4000 if not FAST else 2000, np.random.default_rng(SEED+1))
xg = np.linspace(-3, 3, 500)

fig, ax = plt.subplots(figsize=(6.6, 3.8))
ax.plot(x_tr, y_tr, '.', ms=2.5, color='0.6', label=lab('훈련 자료','training data'))
ax.plot(xg, mu_true(xg), color=CB[0], lw=1.6, label=lab(r'참 $\mu(x)$', r'true $\mu(x)$'))
ax.fill_between(xg, mu_true(xg)-2*sig_true(xg), mu_true(xg)+2*sig_true(xg),
                color=CB[3], alpha=0.18, label=lab(r'참 $\pm 2\sigma(x)$', r'true $\pm 2\sigma(x)$'))
ax.axvline(-1.5, color=CB[4], ls=':', lw=1.4)
ax.axvline(+1.5, color=CB[5], ls=':', lw=1.4)
ax.text(-1.5, 3.1, lab('평균이 어려움','hard mean'), ha='center', fontsize=8, color=CB[4])
ax.text(+1.5, 3.1, lab('잡음이 큼','high noise'), ha='center', fontsize=8, color=CB[5])
ax.set_xlabel('$x$'); ax.set_ylabel('$y$'); ax.set_ylim(-3, 3.5)
ax.set_title(lab('두 종류의 어려움을 서로 다른 위치에 배치했다',
                 'the two difficulties are placed at different locations'), fontsize=10)
ax.legend(fontsize=8, loc='lower right'); show('data')

---
## 2. 망과 두 목적함수

망은 출력이 둘이다 — $\mu_\theta(x)$ 와 $s_\theta(x) = \log\sigma_\theta^2(x)$ (§2.2.2 5절의 매개화).

$$\ell_{\mathrm{MSE}} = r^2, \qquad
\ell_{\mathrm{NLL}} = \tfrac12 s + \tfrac12 r^2 e^{-s}, \qquad r = \mu_\theta(x) - y$$

**등분산 모형은 MSE로 학습한 뒤 §2.2.2 1절의 플러그인 추정 $\hat\sigma^2 = \overline{\mathrm{SE}}$ 를 쓴다.**
그것이 상수 $\sigma$ 에 대한 프로파일 최대우도해이므로, 별도로 최적화할 필요가 없다.

In [ ]:
def init_net(L, w, rng, d_out=1, bstd=0.3):
    dims = [1] + [w]*L + [d_out]; Ws, bs = [], []
    for i in range(len(dims)-1):
        Ws.append(rng.normal(0, np.sqrt(2.0/dims[i]), (dims[i], dims[i+1])))
        bs.append(rng.normal(0, bstd, dims[i+1]) if i < len(dims)-2 else np.zeros(dims[i+1]))
    return Ws, bs

def fwd(x, Ws, bs):
    a = np.asarray(x, float).reshape(-1,1); acts=[a]; pre=[]
    for i in range(len(Ws)-1):
        z = a @ Ws[i] + bs[i]; pre.append(z); a = np.maximum(z,0); acts.append(a)
    return a @ Ws[-1] + bs[-1], acts, pre

def backward(Ws, bs, acts, pre, g):
    gW=[None]*len(Ws); gb=[None]*len(bs)
    gW[-1] = acts[-1].T @ g; gb[-1] = g.sum(0); d = g @ Ws[-1].T
    for i in range(len(Ws)-2, -1, -1):
        d = d*(pre[i] > 0); gW[i] = acts[i].T @ d; gb[i] = d.sum(0)
        if i > 0:
            d = d @ Ws[i].T
    return gW, gb

def train(Ws, bs, x, y, steps, mode, lr=3e-3, s_warm=0):
    st = {'mW':[np.zeros_like(W) for W in Ws], 'vW':[np.zeros_like(W) for W in Ws],
          'mb':[np.zeros_like(b) for b in bs], 'vb':[np.zeros_like(b) for b in bs]}
    n = len(x); hist = []
    for t in range(1, steps+1):
        out, acts, pre = fwd(x, Ws, bs)
        mu = out[:,0]; r = mu - y
        if mode == 'mse':
            hist.append(float(np.mean(r**2)))
            g = np.zeros_like(out); g[:,0] = (2.0/n)*r
        else:
            warm = t <= s_warm
            s = np.zeros_like(r) if warm else np.clip(out[:,1], *S_CLAMP)
            e = np.exp(-s)
            hist.append(float(np.mean(0.5*s + 0.5*r*r*e)))
            g = np.zeros_like(out)
            g[:,0] = (1.0/n)*r*e
            g[:,1] = 0.0 if warm else (0.5/n)*(1.0 - r*r*e)
        gW, gb = backward(Ws, bs, acts, pre, g)
        for i in range(len(Ws)):
            for (p, gp, mk, vk) in ((Ws[i], gW[i], 'mW','vW'), (bs[i], gb[i], 'mb','vb')):
                st[mk][i] = 0.9*st[mk][i] + 0.1*gp
                st[vk][i] = 0.999*st[vk][i] + 0.001*gp**2
                p -= lr*(st[mk][i]/(1-0.9**t))/(np.sqrt(st[vk][i]/(1-0.999**t)) + 1e-8)
    return hist

def test_nll(mu, s, y):
    return float(np.mean(0.5*np.log(2*np.pi) + 0.5*s + 0.5*(y-mu)**2*np.exp(-s)))

def predict(Ws, bs, x):
    o = fwd(x, Ws, bs)[0]
    mu = o[:,0]
    s = np.clip(o[:,1], *S_CLAMP) if o.shape[1] > 1 else None
    return mu, s

# ── 등분산: MSE + 플러그인 σ ──
W_h, b_h = init_net(3, WIDTH, np.random.default_rng(7), d_out=1)
train(W_h, b_h, x_tr, y_tr, STEPS, 'mse')
mu_tr_h = predict(W_h, b_h, x_tr)[0]
s_plug = float(np.log(np.mean((y_tr - mu_tr_h)**2)))
mu_te_h = predict(W_h, b_h, x_te)[0]

# ── 이분산: NLL ──
W_e, b_e = init_net(3, WIDTH, np.random.default_rng(7), d_out=2)
hist_e = train(W_e, b_e, x_tr, y_tr, STEPS, 'nll')
mu_te_e, s_te_e = predict(W_e, b_e, x_te)

print(f"등분산 (MSE + 플러그인 σ̂={np.exp(s_plug/2):.3f})")
print(f"   시험 NLL {test_nll(mu_te_h, s_plug*np.ones_like(mu_te_h), y_te):+.4f}")
print(f"이분산 (NLL)")
print(f"   시험 NLL {test_nll(mu_te_e, s_te_e, y_te):+.4f}")

# 영역별 평균 RMSE
def rmse_region(mu, x, m):
    return float(np.sqrt(np.mean((mu[m] - mu_true(x[m]))**2)))
mL = np.abs(x_te + 1.5) < 0.5      # 평균이 어려운 영역
mR = np.abs(x_te - 1.5) < 0.5      # 잡음이 큰 영역
print(f"\n평균 추정 RMSE      전체      x≈-1.5     x≈+1.5")
print(f"  등분산          {rmse_region(mu_te_h,x_te,np.ones_like(mL,bool)):.4f}   "
      f"{rmse_region(mu_te_h,x_te,mL):.4f}    {rmse_region(mu_te_h,x_te,mR):.4f}")
print(f"  이분산          {rmse_region(mu_te_e,x_te,np.ones_like(mL,bool)):.4f}   "
      f"{rmse_region(mu_te_e,x_te,mL):.4f}    {rmse_region(mu_te_e,x_te,mR):.4f}")

In [ ]:
mu_g_h = predict(W_h, b_h, xg)[0]
mu_g_e, s_g_e = predict(W_e, b_e, xg)
sd_g_e = np.exp(s_g_e/2)

fig, axes = plt.subplots(1, 2, figsize=(10.0, 3.8), sharey=True)
for ax, (m, sd, ttl) in zip(axes, [
        (mu_g_h, np.exp(s_plug/2)*np.ones_like(xg), lab('등분산 (MSE + 플러그인 σ)','homoscedastic')),
        (mu_g_e, sd_g_e, lab('이분산 (NLL)','heteroscedastic'))]):
    ax.plot(x_tr, y_tr, '.', ms=2, color='0.75')
    ax.fill_between(xg, m-2*sd, m+2*sd, color=CB[5], alpha=0.22,
                    label=lab(r'예측 $\pm 2\hat\sigma$', r'predicted $\pm 2\hat\sigma$'))
    ax.plot(xg, mu_true(xg), color=CB[0], lw=1.2, ls='--', label=lab('참 평균','true mean'))
    ax.plot(xg, m, color=CB[5], lw=1.6, label=lab('예측 평균','predicted mean'))
    ax.set_xlabel('$x$'); ax.set_title(ttl, fontsize=10); ax.set_ylim(-3, 3.5)
axes[0].set_ylabel('$y$'); axes[1].legend(fontsize=8, loc='lower right')
fig.suptitle(lab('등분산은 어디서나 같은 폭의 띠를 그린다',
                 'the homoscedastic band has constant width everywhere'), y=1.03, fontsize=10)
show('homo_vs_hetero')

> **이 실행에서는 이분산이 평균 추정에서도 이겼다.** 잡음이 작은 영역의 잔차에 더 큰 가중치가 실려
> 그쪽을 정확히 맞혔기 때문이다. **그러나 이것이 보장되는 것은 아니다** — 6절의 자기 점검 1에서
> 같은 가중이 해가 되는 조건을 묻는다. NLL이 최소화하는 것은 평균의 정확도가 아니라 **분포의 우도**다.

---
## 3. $\hat\sigma(x)$ 가 참 $\sigma(x)$ 를 회복하는가

참값을 알고 있으므로 직접 채점한다. 그리고 **보정**도 확인한다 — 예측 구간 $\pm 2\hat\sigma$ 안에
실제로 몇 %가 들어오는가 (가우시안이면 95.4%여야 한다).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9.6, 3.6))
axes[0].plot(xg, sig_true(xg), color=CB[0], lw=1.8, label=lab(r'참 $\sigma(x)$', r'true $\sigma(x)$'))
axes[0].plot(xg, sd_g_e, color=CB[5], lw=1.8, label=lab(r'추정 $\hat\sigma(x)$', r'estimated'))
axes[0].axhline(np.exp(s_plug/2), color=CB[3], ls='--', lw=1.4,
                label=lab('등분산 상수 $\\hat\\sigma$', 'homoscedastic constant'))
axes[0].set_xlabel('$x$'); axes[0].set_ylabel(r'$\sigma$')
axes[0].set_title(lab('분산 추정', 'variance estimate'), fontsize=10)
axes[0].legend(fontsize=8)

# 보정: 국소 구간별 포함 비율
bins = np.linspace(-3, 3, 13); cen = 0.5*(bins[:-1]+bins[1:])
cov_e, cov_h = [], []
for i in range(len(bins)-1):
    m = (x_te >= bins[i]) & (x_te < bins[i+1])
    cov_e.append(np.mean(np.abs(y_te[m]-mu_te_e[m]) <= 2*np.exp(s_te_e[m]/2)))
    cov_h.append(np.mean(np.abs(y_te[m]-mu_te_h[m]) <= 2*np.exp(s_plug/2)))
axes[1].plot(cen, 100*np.array(cov_e), 'o-', ms=4, color=CB[5], label=lab('이분산','heteroscedastic'))
axes[1].plot(cen, 100*np.array(cov_h), 's-', ms=4, color=CB[3], label=lab('등분산','homoscedastic'))
axes[1].axhline(95.4, color=CB[0], ls=':', lw=1.4, label=lab('목표 95.4%','target 95.4%'))
axes[1].set_xlabel('$x$'); axes[1].set_ylabel(lab(r'$\pm 2\hat\sigma$ 안에 드는 비율 (%)', 'coverage (%)'))
axes[1].set_ylim(50, 105)
axes[1].set_title(lab('국소 보정', 'local calibration'), fontsize=10)
axes[1].legend(fontsize=8)
show('sigma_recovery')

print(f"σ̂(x) 와 참 σ(x) 의 상관: {np.corrcoef(sd_g_e, sig_true(xg))[0,1]:+.3f}")
print(f"\n  x       참 σ    추정 σ̂   등분산 σ̂")
for xv in [-2.0, -1.5, 0.0, 1.5, 2.5]:
    i = int(np.argmin(np.abs(xg - xv)))
    print(f"{xv:+5.1f}   {sig_true(xv):.3f}   {sd_g_e[i]:.3f}    {np.exp(s_plug/2):.3f}")
print(f"\n전체 포함 비율: 이분산 {100*np.mean(np.abs(y_te-mu_te_e)<=2*np.exp(s_te_e/2)):.1f}%  "
      f"등분산 {100*np.mean(np.abs(y_te-mu_te_h)<=2*np.exp(s_plug/2)):.1f}%  (목표 95.4%)")

> **오른쪽 그림이 등분산의 문제를 보여 준다.** 전체 포함 비율은 그럭저럭 맞더라도
> **잡음이 작은 영역에서는 과도하게 넓고 큰 영역에서는 좁습니다.**
> 하나의 집계 수치가 국소적 실패를 감추는 전형적인 예입니다.

---
## 4. 실패 모드 — 모형 오차를 자료 잡음으로 보고한다

§2.2.2 4절에서 유도한 것을 다시 적는다. 최적 분산 $\sigma^2 = r^2$ 에서

$$\ell\big|_{\sigma^2 = r^2} = \log|r| + \text{상수}, \qquad
\left.\frac{\partial\ell}{\partial\mu}\right|_{\sigma^2=r^2} = -\frac{1}{r}$$

**잔차가 클수록 손실 기여가 로그로만 늘고 기울기는 오히려 작아진다.** 그러므로 모형은 어려운 영역에서
$\mu$ 를 고치는 대신 $\sigma$ 를 키워 손실을 회피할 수 있다.

용량을 줄여 가며 **$x \approx -1.5$**(평균이 어렵지만 잡음은 작은 곳)에서 $\hat\sigma$ 가 어떻게 되는지 본다.

In [ ]:
WIDTHS = [4, 8, 16, 64]
res_w = {}
for w in WIDTHS:
    Wc, bc = init_net(3, w, np.random.default_rng(7), d_out=2)
    train(Wc, bc, x_tr, y_tr, STEPS, 'nll')
    mg, sg = predict(Wc, bc, xg)
    res_w[w] = (mg, np.exp(sg/2))

iL = int(np.argmin(np.abs(xg + 1.5)))
iR = int(np.argmin(np.abs(xg - 1.5)))
print("  폭    x=-1.5 σ̂     x=+1.5 σ̂     σ 상관     x=-1.5 평균오차")
print(f"       (참 {sig_true(-1.5):.3f})   (참 {sig_true(1.5):.3f})")
for w in WIDTHS:
    mg, sg = res_w[w]
    mL_ = np.abs(xg + 1.5) < 0.5
    print(f"  {w:>3}     {sg[iL]:.3f}         {sg[iR]:.3f}      {np.corrcoef(sg, sig_true(xg))[0,1]:+.3f}"
          f"      {np.sqrt(np.mean((mg[mL_]-mu_true(xg[mL_]))**2)):.3f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9.8, 3.8))
for w, c in zip(WIDTHS, [CB[1], CB[2], CB[3], CB[5]]):
    axes[0].plot(xg, res_w[w][1], lw=1.5, color=c, label=lab(f'폭 {w}', f'width {w}'))
axes[0].plot(xg, sig_true(xg), color=CB[0], lw=2.2, ls='--', label=lab('참값','truth'))
axes[0].axvline(-1.5, color=CB[4], ls=':', lw=1.2)
axes[0].set_xlabel('$x$'); axes[0].set_ylabel(r'$\hat\sigma(x)$')
axes[0].set_title(lab('용량이 작으면 왼쪽에서도 $\\hat\\sigma$ 가 부푼다',
                      'small capacity inflates $\\hat\\sigma$ on the left too'), fontsize=10)
axes[0].legend(fontsize=8)

mg4 = res_w[4][0]; sg4 = res_w[4][1]
axes[1].plot(x_tr, y_tr, '.', ms=2, color='0.75')
axes[1].fill_between(xg, mg4-2*sg4, mg4+2*sg4, color=CB[1], alpha=0.25,
                     label=lab(r'폭 4의 $\pm 2\hat\sigma$', r'width 4'))
axes[1].plot(xg, mu_true(xg), color=CB[0], lw=1.2, ls='--', label=lab('참 평균','true mean'))
axes[1].plot(xg, mg4, color=CB[1], lw=1.6)
axes[1].set_xlabel('$x$'); axes[1].set_ylim(-3, 3.5)
axes[1].set_title(lab('고주파를 못 맞히고 띠로 덮어 버린다',
                      'it covers the wiggle with a band instead of fitting it'), fontsize=10)
axes[1].legend(fontsize=8)
show('capacity_failure')

> ### 이것이 이 노트북의 핵심 관찰이다
>
> $x \approx -1.5$ 에서 참 $\sigma$ 는 0.08인데, 폭 4인 망은 그보다 몇 배 큰 값을 보고합니다.
> **자료가 시끄러워서가 아니라 모형이 못 맞혀서**입니다.
>
> $$\hat\sigma(x) \ \text{는 우연적 불확실성을 재도록 설계되었지만, 실제로는 모형 오차까지 흡수한다}$$
>
> 그리고 **밖에서 보면 둘을 구별할 수 없습니다.** 참 $\sigma$ 를 모르는 실제 상황에서
> "이 영역은 원래 시끄럽다"와 "우리 모형이 이 영역을 못 배웠다"는 같은 출력으로 나타납니다.
> §50.1이 두 불확실성을 가르고 §50.4가 이 문제를 본격적으로 다룹니다.
>
> **진단 신호:** 용량을 키웠을 때 $\hat\sigma$ 가 줄어드는 영역은 **모형 오차**였던 곳입니다.
> 표의 마지막 열(평균 오차)이 $\hat\sigma$ 와 함께 움직이는 것을 확인하십시오.

---
## 5. $\hat\sigma$ 는 아래로 편향된다

§2.1.5에서 본 것이 여기서도 성립한다. 잔차로 분산을 추정하는데 그 잔차는 $\theta$ 를 자료에 맞춘 뒤의 것이다.
**훈련 자료에서 잰 $\hat\sigma$ 와 시험 자료에서 잰 실제 퍼짐을 비교한다.**

In [ ]:
mu_tr_e, s_tr_e = predict(W_e, b_e, x_tr)
print("           훈련 잔차 RMS   예측 σ̂ 평균   시험 잔차 RMS")
print(f"  이분산      {np.sqrt(np.mean((y_tr-mu_tr_e)**2)):.4f}         "
      f"{np.mean(np.exp(s_tr_e/2)):.4f}        {np.sqrt(np.mean((y_te-mu_te_e)**2)):.4f}")
print(f"  등분산      {np.sqrt(np.mean((y_tr-mu_tr_h)**2)):.4f}         "
      f"{np.exp(s_plug/2):.4f}        {np.sqrt(np.mean((y_te-mu_te_h)**2)):.4f}")
print("\n-> 훈련에서 잰 퍼짐이 시험보다 작다. σ̂ 은 그 훈련 잔차에 맞춰졌으므로 함께 작다.")
print("   예측 구간이 좁게 나오며, 이것이 §50.6에서 보정이 필요한 이유의 하나다.")

---
## 6. 자기 점검

1. 아래 셀이 보이듯 평균 학습의 실효 가중치가 두 영역에서 30배 넘게 차이 난다. **어떤 상황에서 이것이 해가 되는가?**
2. 4절에서 폭을 키우면 $x\approx-1.5$ 의 $\hat\sigma$ 가 줄었다. **참 $\sigma$ 를 모르는 상황에서** 이 진단을 어떻게 쓰겠는가?
3. $\sigma$ 를 학습하지 않고 고정하면(등분산) 4절의 실패가 사라지는가? 대신 무엇을 잃는가?
4. 3절의 전체 포함 비율은 두 모형에서 비슷할 수 있다. **그럼에도 등분산이 나쁜 이유**를 국소 그림으로 설명하라.

In [ ]:
# 자기 점검 1의 확인 — 가중치가 어디에 실리는가
mu_c, s_c = predict(W_e, b_e, x_tr)
wgt = np.exp(-s_c)                       # 1/σ²(x) : 평균 학습의 실효 가중치
mL_ = np.abs(x_tr + 1.5) < 0.5
mR_ = np.abs(x_tr - 1.5) < 0.5
print("평균 학습에 실리는 실효 가중치 1/σ̂²(x)")
print(f"  x≈-1.5 (잡음 작음): {wgt[mL_].mean():8.2f}")
print(f"  x≈+1.5 (잡음 큼)  : {wgt[mR_].mean():8.2f}")
print(f"  비                : {wgt[mL_].mean()/wgt[mR_].mean():8.1f} 배")
print("\n-> 이분산 모형은 잡음이 큰 영역의 평균을 거의 학습하지 않는다.")
print("   그 영역의 평균 RMSE가 나빠지는 것은 버그가 아니라 목적함수가 그렇게 시킨 것이다.")
print("   '평균을 잘 맞히는 것'이 목표라면 NLL이 그 목표가 아니라는 뜻이기도 하다.")

---
## 7. 직접 바꿔 볼 손잡이

| 손잡이 | 위치 | 기본값 | 바꾸면 |
|---|---|---|---|
| `WIDTH` | 0절 | 64 | 주 모형의 용량. 8로 낮추면 4절의 실패가 주 결과에도 나타난다 |
| `WIDTHS` | 4절 | [4,8,16,64] | 용량 훑기 |
| `N_TRAIN` | 0절 | 1500 | 표본 수. 줄이면 $\hat\sigma$ 의 편향(5절)이 커진다 |
| `S_CLAMP` | 0절 | (−8, 8) | $\log\sigma^2$ 절단. 넓히면 §2.1.6의 발산에 가까워진다 |
| `s_warm` | `train` 인자 | 0 | 초기에 $\sigma$ 를 고정해 두는 걸음 수 (§2.2.2 6절 (a)) |
| 잡음 위치 | 1절 | $+1.5$ | 잡음 봉우리와 고주파 봉우리를 **같은 곳**에 두면 진단이 불가능해진다 |

**권하는 첫 실험** — 1절에서 고주파 항의 중심을 $-1.5$ 에서 $+1.5$ 로 옮겨 **두 어려움을 겹치십시오.**
그러면 $\hat\sigma$ 가 크게 나오는 것이 잡음 때문인지 모형 오차 때문인지 **원리적으로 판정할 수 없게** 됩니다.
이 노트북이 진단할 수 있었던 것은 설계가 둘을 갈라 놓았기 때문이며, **실제 자료에서는 갈라져 있지 않습니다.**

In [ ]:
print(f"총 실행 시간: {time.time() - _t0:.1f}초")